# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 colorectal cancer dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We'll load the dataset from its Croissant JSON-LD schema, review the record sets and fields using their `@id`, extract tables into Pandas DataFrames, and perform exploratory data analysis and basic visualization.

### Dataset Source
Schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate Dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (do not subscript as dict)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")


## 2. Data Overview
Review available record sets, their fields, columns, and `@id` attributes.

All `@id` fields uniquely identify entities within the Croissant schema. Let's explore the dataset's structure and IDs for each record set and its fields.

In [ ]:
# List all record sets and print their @id and field @id

all_record_sets = list(dataset.record_sets)
print("Record sets in the dataset:")
for rs in all_record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '[Unnamed]')}")
    # Fields for each RecordSet
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # Ensure list
    print(f"  Fields:")
    for fld in fields:
        if isinstance(fld, dict):
            print(f"    - Field @id: {fld.get('@id')}, Name: {fld.get('name')}")
        else:
            print(f"    - Field @id: {fld}")

For illustration, let's enumerate the first record set's columns.


In [ ]:
# If there is at least one record set, list all columns (@id) for it
if all_record_sets:
    primary_rs = all_record_sets[0]
    print(f"\nPrimary RecordSet @id: {primary_rs['@id']}")
    # List columns by @id (from field definitions)
    for field in primary_rs.get('field', []):
        if isinstance(field, dict):
            print(f"Field @id: {field.get('@id')}, name: {field.get('name')}, dataType: {field.get('dataType')}")
        else:
            print(f"Field reference: {field}")

## 3. Data Extraction
We use the RecordSet and field `@id`s from the overview to extract tables.

Let's load records from all record sets into pandas DataFrames indexed by `@id`.


In [ ]:
# Extract all record sets into DataFrames, using @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in all_record_sets]

for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    if records:
        dataframes[recset_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[recset_id])} records for RecordSet @id: {recset_id}")
    else:
        print(f"No records found for RecordSet @id: {recset_id}")

Now, let's preview the first DataFrame (if available):

In [ ]:
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"DataFrame Columns for RecordSet @id: {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field and perform basic filtering, normalization, and grouping.

All fields are referenced by their `@id`. We'll choose a column appearing numeric in the preview above, or you can substitute the relevant `@id` from the record set schema. Please refer to the previous code's printout for available field `@id`s.

In [ ]:
# Example: assuming the dataset includes 'Age' by its field @id
# Please adapt numeric_field_id and group_field_id to your dataset as needed.

# Example (replace with real @id from your schema):
# Let's search for a field that has 'age' in its name from the columns

import re

numeric_field_id = None
group_field_id = None

if dataframes:
    # Use first record set for exploration
    rs_id = first_rs_id
    cols = dataframes[rs_id].columns.tolist()
    for col in cols:
        if re.search('age', str(col), re.IGNORECASE):
            numeric_field_id = col
        if re.search('sex|gender', str(col), re.IGNORECASE):
            group_field_id = col
    # Fallbacks
    if not numeric_field_id and cols:
        numeric_field_id = cols[0]
    if not group_field_id and len(cols) > 1:
        group_field_id = cols[1]
    print(f"Selected numeric field @id: {numeric_field_id}")
    print(f"Selected group field @id: {group_field_id}")
    # Run EDA
    try:
        # Convert the selected field to numeric if possible
        df = dataframes[rs_id].copy()
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].quantile(0.25)  # e.g., 25th percentile as a threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (25th percentile):")
        print(filtered_df[[numeric_field_id, group_field_id]].head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head())

        # Group by group_field_id and compute mean of numeric_field_id
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    except Exception as e:
        print(f"EDA Error: {e}")

## 5. Visualization
Visualize the distribution of the selected numeric field and groupings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[first_rs_id].copy()
    if numeric_field_id and numeric_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=10, kde=True)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
    if group_field_id and group_field_id in df.columns and numeric_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded the FAIR^2 colorectal cancer survivors dataset via its Croissant schema using `mlcroissant`.
- Explored record sets and fields via their `@id`.
- Extracted tabular data for analysis, filtered and normalized a numeric attribute (such as `Age`), and grouped records by categorical attributes (such as `Sex`).
- Visualized distributions and groupwise comparisons for initial insight.

**For further analysis, refer to the dataset documentation and the Croissant schema to explore the rich set of variables (`@id` fields) and relationships.**